In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
# pip install seaborn
import seaborn as sns

In [ ]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_roubos = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')
print(df_roubos.head())
# print(df_roubos.describe())
# print(df_roubos['regiao'].unique()) # é usado para verificar os valores únicos da coluna 'regiao' do DataFrame df_roubos. 
# Isso ajuda a entender quais regiões estão presentes nos dados e pode ser útil para análise ou filtragem posterior.

In [11]:
# delimitar cisp, regiao, munic e roubo_veiculo
df_roubos = df_roubos[['cisp', 'regiao', 'munic', 'roubo_veiculo']]

# corrigindo a Grande Niterói 
correcao_acentuacao = df_roubos['regiao'].str.startswith('Grande Niter', na=False)
# print(correcao_acentuacao)
df_roubos.loc[correcao_acentuacao, 'regiao'] = 'Grande Niterói'

print(df_roubos['regiao'].unique())

<StringArray>
['Capital', 'Baixada Fluminense', 'Interior', 'Grande Niterói']
Length: 4, dtype: str


In [16]:
# agrupando e totalizando os roubos de veículos por cisp, região e município
df_roubos_agrupados = df_roubos.groupby(['cisp', 'regiao', 'munic'], as_index=False)['roubo_veiculo'].sum()
print(df_roubos_agrupados)
# display(df_roubos_agrupados()) não funciona no vscode

     cisp    regiao           munic  roubo_veiculo
0       1   Capital  Rio de Janeiro            583
1       4   Capital  Rio de Janeiro           2230
2       5   Capital  Rio de Janeiro           1384
3       6   Capital  Rio de Janeiro           4708
4       7   Capital  Rio de Janeiro           2035
..    ...       ...             ...            ...
142   165  Interior     Mangaratiba            504
143   166  Interior  Angra dos Reis            971
144   167   Capital          Paraty              2
145   167  Interior          Paraty            106
146   168  Interior       Rio Claro             66

[147 rows x 4 columns]


In [17]:
# calculando a média, mediana, maximo, minimo e quartis 

array_roubos = np.array(df_roubos_agrupados['roubo_veiculo'])

media = np.mean(array_roubos)
mediana = np.median(array_roubos)

q1 = np.percentile(array_roubos, 25)
q3 = np.percentile(array_roubos, 75)

maximo = np.max(array_roubos)
minimo = np.min(array_roubos)

print(f'Média: {media}')
print(f'Mediana: {mediana}')
print(f'Mínimo: {minimo}')
print(f'Máximo: {maximo}')
print(f'1º Quartil (Q1): {q1}')
print(f'3º Quartil (Q3): {q3}')


Média: 4969.843537414966
Mediana: 942.0
Mínimo: 2
Máximo: 38084
1º Quartil (Q1): 90.0
3º Quartil (Q3): 7390.0


In [31]:
# criando uma série com os maiores 

df_maiores = df_roubos_agrupados[df_roubos_agrupados['roubo_veiculo'] >= q3].copy()
df_maiores['flag'] = 'mais'
print(df_maiores)

df_menores = df_roubos_agrupados[df_roubos_agrupados['roubo_veiculo'] <= q1].copy()
df_menores['flag'] = 'menos'
print(df_menores)

    cisp              regiao               munic  roubo_veiculo  flag
13    17             Capital      Rio de Janeiro           7771  mais
14    18             Capital      Rio de Janeiro           7393  mais
16    20             Capital      Rio de Janeiro           9201  mais
17    21             Capital      Rio de Janeiro          19884  mais
18    22             Capital      Rio de Janeiro          15236  mais
19    23             Capital      Rio de Janeiro          10833  mais
20    24             Capital      Rio de Janeiro          12311  mais
21    25             Capital      Rio de Janeiro          11140  mais
22    26             Capital      Rio de Janeiro           7920  mais
23    27             Capital      Rio de Janeiro          28171  mais
24    28             Capital      Rio de Janeiro          11120  mais
25    29             Capital      Rio de Janeiro          17260  mais
26    30             Capital      Rio de Janeiro          12987  mais
27    31            

In [29]:
# concatenando os dois dataframes df_maiores e df_menores em um único dataframe df_final

df_final = pd.concat([df_maiores, df_menores], ignore_index=True)
print(df_final)

    cisp    regiao                 munic  roubo_veiculo   flag
0     17   Capital        Rio de Janeiro           7771   mais
1     18   Capital        Rio de Janeiro           7393   mais
2     20   Capital        Rio de Janeiro           9201   mais
3     21   Capital        Rio de Janeiro          19884   mais
4     22   Capital        Rio de Janeiro          15236   mais
..   ...       ...                   ...            ...    ...
69   156  Interior  Santa Maria Madalena             47  menos
70   157  Interior     Trajano de Morais             21  menos
71   158  Interior            Bom Jardim             21  menos
72   167   Capital                Paraty              2  menos
73   168  Interior             Rio Claro             66  menos

[74 rows x 5 columns]


In [30]:
# exportando os dados utf-8-sig para ler no powerbi

df_final.to_csv('roubos_veiculos.csv', index=False, encoding='utf-8-sig')